# 🏥 RAG Oncologie — Version Corrigée (clés JSON réelles)
Toutes les clés de chunking correspondent à la vraie structure du fichier `merged_cancers_vfrancais.json`.

## Installation

In [4]:
!pip install -q sentence-transformers rank-bm25 pandas numpy scikit-learn transformers torch
!pip install -q langchain-google-vertexai
print('✅ Installation terminée')

✅ Installation terminée


##  Imports

In [6]:
import json
import pickle
import re
import time
import warnings
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from collections import Counter
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from sklearn.metrics import ndcg_score

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from datasets import Dataset

# RAGAS désactivé temporairement
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision

warnings.filterwarnings('ignore')

print("✅ Imports réussis")
print(f"Torch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

✅ Imports réussis
Torch version: 2.9.1+cpu
CUDA disponible: False


## 1. Chargement et normalisation

In [7]:
# Dictionnaires de normalisation
PREFIX_TO_CANCER = {
    'SCLC': 'Cancer du poumon à petites cellules',
    'CPNPC': 'Cancer Pulmonaire Non à Petites Cellules (CPNPC)',
    'BREAST': 'Cancer du sein',
    'CRC': 'Cancer Colorectal (CCR)',
    'PROSTATE': 'Cancer de la prostate',
    'THYROID': 'Cancer de la thyroïde',
    'OVARIAN': "Cancer épithélial de l'ovaire",
    'MEL': 'Mélanome',
    'HGG': 'Gliomes de haut grade',
    'STOMACH': 'Cancer gastrique',
    'KIDNEY': 'Cancer du rein',
    'BLADDER': 'Cancer de la vessie',
    'CERVICAL': "Cancer du col de l'utérus",
}

NAME_NORMALIZATION = {
    'Thyroid Cancer': 'Cancer de la thyroïde',
    'Prostate Cancer': 'Cancer de la prostate',
    'Kidney Cancer (Renal Cell Carcinoma)': 'Cancer du rein',
    'Gastric Cancer (Gastric Adenocarcinoma)': 'Cancer gastrique',
    'Gastric Cancer': 'Cancer gastrique',
    'Breast Cancer': 'Cancer du sein',
    'Ovarian Cancer': "Cancer épithélial de l'ovaire",
    'Colorectal Cancer': 'Cancer Colorectal (CCR)',
    'Melanoma': 'Mélanome',
    'Glioma': 'Gliomes de haut grade',
    'Small Cell Lung Cancer': 'Cancer du poumon à petites cellules',
}

# Chargement
with open('merged_cancers_vfrancais.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
print(f'✅ {len(data)} documents chargés')

# Normalisation
fixed = 0
for doc in data:
    name = doc.get('cancer_name', '') or ''
    if name in NAME_NORMALIZATION:
        doc['cancer_name'] = NAME_NORMALIZATION[name]
        fixed += 1
    elif not name or name == 'unknown':
        prefix = doc.get('document_id', '').split('_')[0]
        cancer = PREFIX_TO_CANCER.get(prefix)
        if cancer:
            doc['cancer_name'] = cancer
            fixed += 1

counts = Counter(d.get('cancer_name', 'unknown') for d in data)
print(f'✅ {fixed} documents normalisés')
print(f'📊 {len(counts)} types de cancers différents')

✅ 233 documents chargés
✅ 150 documents normalisés
📊 16 types de cancers différents


## 2. Chunking avec les vraies clés JSON

In [9]:
def safe_string(val, maxlen=400):
    """Convertit n'importe quelle valeur en texte lisible."""
    if val is None or val == '' or val == []:
        return ''
    if isinstance(val, list):
        parts = [safe_string(v, maxlen) for v in val if v]
        return '; '.join(p for p in parts if p)[:maxlen]
    if isinstance(val, dict):
        parts = [f"{k}: {safe_string(v, 100)}" for k, v in val.items() if v]
        return ' | '.join(parts)[:maxlen]
    return str(val)[:maxlen]

def chunk_document(item: Dict) -> List[Dict]:
    chunks = []
    cancer = item.get('cancer_name', 'unknown')
    doc_type = item.get('document_type', 'unknown')
    doc_id = item.get('document_id', 'unknown')

    def add(text: str, subtype: str):
        text = text.strip()
        if len(text) > 60:
            chunks.append({
                'text': f'[{cancer}] [{subtype}] {text[:700]}',
                'metadata': {'cancer': cancer, 'type': doc_type, 'subtype': subtype, 'doc_id': doc_id}
            })

    # treatment_protocol
    if doc_type == 'treatment_protocol':
        for p in item.get('protocols', []):
            drugs_str = safe_string([f"{d.get('drug_name','?')} {d.get('dose','')}" for d in p.get('drugs', [])])
            add(
                f"Protocole: {p.get('protocol_name','N/A')}. "
                f"Cancer: {p.get('cancer_type','')}. "
                f"Stade: {p.get('stage','')}. "
                f"Ligne: {p.get('line_of_therapy','')}. "
                f"Médicaments: {drugs_str}. "
                f"Résultats: {safe_string(p.get('expected_outcomes',[]))}",
                'traitement_protocole'
            )

    # cancer_knowledge
    elif doc_type == 'cancer_knowledge':
        field_map = {
            'definition': 'Définition',
            'epidemiology': 'Épidémiologie',
            'risk_factors': 'Facteurs de risque',
            'common_symptoms': 'Symptômes',
            'prognosis': 'Pronostic',
        }
        for key, label in field_map.items():
            val = item.get(key)
            if val:
                add(f'{label}: {safe_string(val)}', f'connaissance_{key}')

    # metastasis
    elif doc_type == 'metastasis':
        for p in item.get('metastatic_profiles', []):
            add(
                f"Métastase de {p.get('primary_cancer','?')} vers {p.get('metastatic_site','?')}. "
                f"Traitement: {safe_string(p.get('treatment_options',[]))}",
                'metastase'
            )

    # toxicity_management
    elif doc_type == 'toxicity_management':
        for p in item.get('toxicity_profiles', []):
            add(
                f"Toxicité: {p.get('toxicity_name','?')}. "
                f"Médicaments: {safe_string(p.get('causing_drugs',[]))}. "
                f"Gestion: {safe_string(p.get('management_by_grade',{}))}",
                'toxicite'
            )

    # resistance_mechanism
    elif doc_type == 'resistance_mechanism':
        for p in item.get('resistance_profiles', []):
            add(
                f"Résistance: {p.get('drug_name','?')}. "
                f"Mécanisme: {p.get('mechanism','')}. "
                f"Alternatives: {safe_string(p.get('alternative_treatments',[]))}",
                'resistance'
            )

    # oncology_emergency
    elif doc_type == 'oncology_emergency':
        for p in item.get('oncology_emergencies', []):
            add(
                f"Urgence: {p.get('emergency_name','?')}. "
                f"Actions: {safe_string(p.get('immediate_actions',[]))}. "
                f"Traitement: {safe_string(p.get('recommended_treatment',[]))}",
                'urgence'
            )

    # drug
    elif doc_type in ('drug', 'drug_profile_standard'):
        for p in item.get('drug_profiles', []):
            add(
                f"Médicament: {p.get('drug_name','?')}. "
                f"Classe: {p.get('drug_class','')}. "
                f"Indications: {safe_string(p.get('approved_indications',[]))}",
                'medicament'
            )

    # followup
    elif doc_type == 'followup':
        for p in item.get('followup_protocols', []):
            add(
                f"Suivi — Cancer: {p.get('cancer_type','?')}. "
                f"Fréquence: {p.get('followup_frequency','')}. "
                f"Tests: {safe_string(p.get('recommended_tests',[]))}",
                'suivi'
            )

    # staging_system
    elif doc_type == 'staging_system':
        for p in item.get('staging_systems', []):
            add(
                f"Stadification: {p.get('staging_system','')}. "
                f"Description: {p.get('system_description','')}",
                'stadification'
            )

    return chunks

# Création des chunks
all_chunks = []
for doc in data:
    all_chunks.extend(chunk_document(doc))

chunk_types = Counter(c['metadata']['type'] for c in all_chunks)
print(f'✅ {len(all_chunks)} chunks créés')
print('\n📊 Chunks par type:')
for t, n in chunk_types.most_common(10):
    print(f'   {t}: {n}')

# Sauvegarde
with open('chunks_v2.json', 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)
print('\n💾 chunks_v2.json sauvegardé')

✅ 464 chunks créés

📊 Chunks par type:
   cancer_knowledge: 70
   drug: 66
   toxicity_management: 54
   treatment_protocol: 51
   oncology_emergency: 49
   resistance_mechanism: 46
   metastasis: 45
   followup: 43
   staging_system: 32
   drug_profile_standard: 8

💾 chunks_v2.json sauvegardé


## 3. Embeddings 

In [10]:
MODEL_NAME = 'intfloat/multilingual-e5-small'
print(f'🔄 Chargement {MODEL_NAME}...')

model = SentenceTransformer(MODEL_NAME)
print(f'✅ Modèle chargé (dim: {model.get_sentence_embedding_dimension()})')

def encode_passages(texts):
    return model.encode([f'passage: {t}' for t in texts],
                        show_progress_bar=True, 
                        convert_to_numpy=True,
                        normalize_embeddings=True)

def encode_query(query):
    return model.encode([f'query: {query}'], 
                        convert_to_numpy=True,
                        normalize_embeddings=True)

print(f'\n📝 Encodage de {len(all_chunks)} passages...')
embeddings = encode_passages([c['text'] for c in all_chunks])
print(f'✅ Embeddings: {embeddings.shape}')

with open('embeddings_v2.pkl', 'wb') as f:
    pickle.dump(embeddings, f)
print('💾 embeddings_v2.pkl sauvegardé')

🔄 Chargement intfloat/multilingual-e5-small...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1454.41it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modèle chargé (dim: 384)

📝 Encodage de 464 passages...


Batches: 100%|██████████| 15/15 [00:59<00:00,  3.97s/it]

✅ Embeddings: (464, 384)
💾 embeddings_v2.pkl sauvegardé


## 4. Retriever hybride avec reranking

In [17]:
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity

class SmartRetriever:
    """
    Retriever intelligent avec :
    - Détection sémantique du type de question (via embeddings)
    - Boost adaptatif par type de document
    - Re-ranking par similarité contextuelle
    - Fusion multi-stratégies
    """
    
    def __init__(self, chunks, embeddings, encode_query_fn):
        self.chunks = chunks
        self.embeddings = embeddings
        self.encode_query_fn = encode_query_fn
        
        # BM25
        tokenized = [self._tok(c['text']) for c in chunks]
        self.bm25 = BM25Okapi(tokenized)
        
        # Mapping des types pour boost
        self.type_boost_map = {
            # Traitement
            'treatment_protocol': {'traitement': 2.5, 'general': 1.5},
            'drug': {'traitement': 1.8, 'general': 1.2},
            'clinical_case': {'traitement': 1.5},
            
            # Effets secondaires
            'toxicity_management': {'effets_secondaires': 3.0, 'general': 1.3},
            'contraindication_interaction': {'effets_secondaires': 2.0},
            
            # Métastases
            'metastasis': {'metastase': 3.0, 'general': 1.4},
            
            # Urgences
            'oncology_emergency': {'urgence': 3.0, 'general': 1.5},
            
            # Diagnostic
            'diagnostic_guideline': {'diagnostic': 2.5, 'general': 1.3},
            'biomarker_genetics': {'diagnostic': 2.0},
            'pathology': {'diagnostic': 1.8},
            
            # Pronostic
            'staging_system': {'pronostic': 2.0, 'general': 1.2},
            'cancer_knowledge': {'pronostic': 1.5, 'general': 1.1},
            
            # Suivi
            'followup': {'suivi': 2.5, 'general': 1.2},
            
            # Résistance
            'resistance_mechanism': {'resistance': 2.5, 'general': 1.3},
        }
        
        print(f'✅ SmartRetriever initialisé ({len(chunks)} chunks)')
    
    def _tok(self, text):
        return [t for t in re.sub(r'[^\w\s]', ' ', text.lower()).split() if len(t) > 1]
    
    def _normalize(self, arr):
        mn, mx = arr.min(), arr.max()
        if mx - mn < 1e-10:
            return np.zeros_like(arr)
        return (arr - mn) / (mx - mn)
    
    def detect_cancer_smart(self, query: str) -> str:
        """Détection intelligente du cancer"""
        q = query.lower()
        # Exact matches first
        for alias, cancer in CANCER_ALIASES.items():
            if alias in q:
                return cancer
        # Partial matches
        for cancer in KNOWN_CANCERS:
            if cancer.lower() in q:
                return cancer
        return None
    
    def detect_qtype_smart(self, query: str) -> str:
        """Détection sémantique du type de question"""
        q = query.lower()
        
        # Patterns avancés
        patterns = {
            'traitement': [
                r'traitement|thérapie|protocole|médicament|chimioth|immunoth|radioth',
                r'quel est le traitement|comment traiter|prise en charge'
            ],
            'effets_secondaires': [
                r'effet.?secondaire|toxicit|indésirable|nausée|vomissement|fatigue',
                r'effets secondaires|effets indésirables|tolérance'
            ],
            'metastase': [
                r'métastase|métastatique|dissémination|localisation secondaire',
                r'métastases cérébrales|métastases osseuses'
            ],
            'urgence': [
                r'urgence|compression médullaire|hypercalcémie|aplasie|syndrome',
                r'urgence oncologique|complication aiguë'
            ],
            'diagnostic': [
                r'diagnostic|dépistage|marqueur|biomarqueur|examen|imagerie',
                r'comment diagnostiquer|quel examen'
            ],
            'pronostic': [
                r'pronostic|survie|récidive|évolution|espérance de vie',
                r'quel est le pronostic|facteur pronostique'
            ],
            'suivi': [
                r'suivi|surveillance|post-traitement|après traitement',
                r'comment suivre|fréquence de suivi'
            ],
            'resistance': [
                r'résistance|mécanisme de résistance|acquise|primaire',
                r'résistance au traitement|échappement'
            ]
        }
        
        scores = {}
        for qtype, pattern_list in patterns.items():
            score = 0
            for pattern in pattern_list:
                if re.search(pattern, q, re.IGNORECASE):
                    score += 1
            if score > 0:
                scores[qtype] = score
        
        if scores:
            return max(scores, key=scores.get)
        return 'general'
    
    def search(self, query, k=5, cancer_filter=None, qtype=None):
        """
        Recherche intelligente avec boost adaptatif
        """
        # 1. Détection automatique
        if cancer_filter is None:
            cancer_filter = self.detect_cancer_smart(query)
        if qtype is None:
            qtype = self.detect_qtype_smart(query)
        
        # 2. Calcul des scores bruts
        bm25_raw = self.bm25.get_scores(self._tok(query))
        sem_raw = np.dot(self.embeddings, self.encode_query_fn(query).T).flatten()
        
        # 3. Normalisation
        bm25_norm = self._normalize(bm25_raw)
        sem_norm = self._normalize(sem_raw)
        
        # 4. Fusion hybride avec cohérence
        coherence = bm25_norm * sem_norm
        hybrid = 0.35 * bm25_norm + 0.50 * sem_norm + 0.15 * coherence
        
        # 5. Application du boost par type de document
        boosted_scores = hybrid.copy()
        for idx in range(len(self.chunks)):
            doc_type = self.chunks[idx]['metadata'].get('type', 'unknown')
            boosts = self.type_boost_map.get(doc_type, {})
            
            # Boost selon le type de question
            boost = boosts.get(qtype, boosts.get('general', 1.0))
            boosted_scores[idx] = hybrid[idx] * boost
            
            # Bonus si le cancer correspond exactement
            doc_cancer = self.chunks[idx]['metadata'].get('cancer', '')
            if cancer_filter and doc_cancer == cancer_filter:
                boosted_scores[idx] *= 1.2
        
        # 6. Tri et filtrage
        indices = np.argsort(boosted_scores)[::-1]
        
        results = []
        seen_texts = set()
        
        for idx in indices:
            if len(results) >= k:
                break
                
            c = self.chunks[idx]
            doc_cancer = c['metadata'].get('cancer', 'unknown')
            doc_type = c['metadata'].get('type', 'unknown')
            
            # Filtre cancer (strict ou non selon confiance)
            if cancer_filter:
                if cancer_filter.lower() not in doc_cancer.lower():
                    # Fallback: si aucun résultat, on relâche le filtre
                    if len(results) == 0 and idx == indices[0]:
                        pass  # Premier résultat accepté même si cancer différent
                    else:
                        continue
            
            # Éviter les doublons trop similaires
            text_preview = c['text'][:100]
            if text_preview in seen_texts:
                continue
            seen_texts.add(text_preview)
            
            results.append({
                'text': c['text'],
                'score': float(boosted_scores[idx]),
                'score_hybrid': float(hybrid[idx]),
                'cancer': doc_cancer,
                'type': doc_type,
                'subtype': c['metadata'].get('subtype', ''),
                'qtype_detected': qtype,
            })
        
        return results
    
    def search_auto(self, query, k=5):
        """Recherche automatique avec détection intelligente"""
        return self.search(query, k=k)

# Initialisation
smart_retriever = SmartRetriever(all_chunks, embeddings, encode_query)
print('\n✅ SmartRetriever prêt !')

# Test
print("\n" + "="*80)
print("🧠 TEST DU SMART RETRIEVER")
print("="*80)

test_queries = [
    "Quel est le traitement du cancer de la prostate ?",
    "Effets secondaires de la chimiothérapie cancer du poumon",
    "Comment diagnostiquer le cancer de l'ovaire ?",
    "Métastases cérébrales du mélanome",
    "Urgence compression médullaire cancer",
]

for q in test_queries:
    print(f"\n❓ {q}")
    cancer = smart_retriever.detect_cancer_smart(q)
    qtype = smart_retriever.detect_qtype_smart(q)
    print(f"   🎯 Détection: Cancer={cancer}, Type={qtype}")
    
    results = smart_retriever.search(q, k=3)
    for i, r in enumerate(results):
        icon = "💊" if "treatment" in r['type'] else "⚠️" if "toxicity" in r['type'] else "🚨" if "emergency" in r['type'] else "📄"
        print(f"   {icon} [{i+1}] {r['type']}: {r['score']:.3f}")
        print(f"       {r['text'][:120]}...")
    print("-"*70)

✅ SmartRetriever initialisé (464 chunks)

✅ SmartRetriever prêt !

🧠 TEST DU SMART RETRIEVER

❓ Quel est le traitement du cancer de la prostate ?
   🎯 Détection: Cancer=Cancer de la prostate, Type=traitement
   💊 [1] treatment_protocol: 2.356
       [Cancer de la prostate] [traitement_protocole] Protocole: Metastatic Castration-Resistant Prostate Cancer (mCRPC) — Syst...
   💊 [2] treatment_protocol: 2.277
       [Cancer de la prostate] [traitement_protocole] Protocole: Metastatic Castration-Sensitive Prostate Cancer (mCSPC) — Firs...
   💊 [3] treatment_protocol: 2.243
       [Cancer de la prostate] [traitement_protocole] Protocole: Very Low-Risk and Low-Risk Localized Prostate Cancer — Active ...
----------------------------------------------------------------------

❓ Effets secondaires de la chimiothérapie cancer du poumon
   🎯 Détection: Cancer=Cancer Pulmonaire Non à Petites Cellules (CPNPC), Type=traitement
   💊 [1] treatment_protocol: 2.452
       [Cancer Pulmonaire Non à Petites

In [18]:
class FixedSmartRetriever(SmartRetriever):
    """Version corrigée avec meilleure détection des types"""
    
    def detect_qtype_smart(self, query: str) -> str:
        """Détection sémantique améliorée"""
        q = query.lower()
        
        # Ordre important: les plus spécifiques d'abord
        patterns = {
            'effets_secondaires': [
                r'effets? secondaires?', r'effet secondaire', r'toxicite', r'toxicité',
                r'effets indésirables', r'nausée', r'vomissement', r'fatigue',
                r'alopécie', r'neutropénie', r'anémie', r'thrombopénie',
                r'neuropathie', r'cardiaque', r'hépatique', r'rénal'
            ],
            'metastase': [
                r'métastase', r'métastatique', r'métastases? cérébrales?',
                r'métastases? osseuses?', r'dissémination', r'localisation secondaire'
            ],
            'urgence': [
                r'urgence', r'compression médullaire', r'hypercalcémie',
                r'aplasie fébrile', r'syndrome de lyse', r'urgences oncologiques'
            ],
            'diagnostic': [
                r'diagnostic', r'dépistage', r'marqueur', r'biomarqueur',
                r'examen', r'imagerie', r'biopsie', r'comment diagnostiquer',
                r'quel examen', r'quelle imagerie'
            ],
            'pronostic': [
                r'pronostic', r'survie', r'récidive', r'évolution',
                r'espérance de vie', r'facteur pronostique'
            ],
            'suivi': [
                r'suivi', r'surveillance', r'post-traitement',
                r'après traitement', r'comment suivre'
            ],
            'resistance': [
                r'résistance', r'mécanisme de résistance', r'acquise',
                r'primaire', r'résistance au traitement'
            ],
            'traitement': [
                r'traitement', r'thérapie', r'protocole', r'médicament',
                r'chimiothérapie', r'immunothérapie', r'radiothérapie',
                r'chirurgie', r'prise en charge'
            ],
        }
        
        # Scoring pondéré
        scores = {}
        for qtype, pattern_list in patterns.items():
            score = 0
            for pattern in pattern_list:
                if re.search(pattern, q, re.IGNORECASE):
                    # Bonus pour correspondance exacte
                    if len(pattern) > 10 and pattern in q:
                        score += 2
                    else:
                        score += 1
            if score > 0:
                scores[qtype] = score
        
        # Debug
        if scores:
            best = max(scores, key=scores.get)
            print(f"   🔍 Debug: scores={scores} -> best={best}")
            return best
        
        return 'general'

# Recréer le retriever avec la correction
fixed_retriever = FixedSmartRetriever(all_chunks, embeddings, encode_query)

print("\n" + "="*80)
print("🧪 TEST AVEC DÉTECTION CORRIGÉE")
print("="*80)

test_cases = [
    "Effets secondaires de la chimiothérapie cancer du poumon",
    "Quels sont les effets secondaires du cisplatine ?",
    "Traitement du cancer du sein métastatique",
    "Diagnostic du cancer de l'ovaire",
]

for q in test_cases:
    print(f"\n❓ {q}")
    qtype = fixed_retriever.detect_qtype_smart(q)
    print(f"   ✅ Type détecté: {qtype}")
    
    results = fixed_retriever.search(q, k=3)
    for i, r in enumerate(results):
        icon = "⚠️" if "toxicity" in r['type'] else "💊" if "treatment" in r['type'] else "📄"
        print(f"   {icon} [{i+1}] {r['type']}: {r['score']:.3f}")
        print(f"       {r['text'][:100]}...")
    print("-"*50)

✅ SmartRetriever initialisé (464 chunks)

🧪 TEST AVEC DÉTECTION CORRIGÉE

❓ Effets secondaires de la chimiothérapie cancer du poumon
   🔍 Debug: scores={'effets_secondaires': 1, 'traitement': 3} -> best=traitement
   ✅ Type détecté: traitement
   🔍 Debug: scores={'effets_secondaires': 1, 'traitement': 3} -> best=traitement
   💊 [1] treatment_protocol: 2.452
       [Cancer Pulmonaire Non à Petites Cellules (CPNPC)] [traitement_protocole] Protocole: Cisplatine + Pe...
   💊 [2] treatment_protocol: 1.860
       [Cancer Pulmonaire Non à Petites Cellules (CPNPC)] [traitement_protocole] Protocole: Carboplatine + ...
   💊 [3] treatment_protocol: 1.785
       [Cancer Pulmonaire Non à Petites Cellules (CPNPC)] [traitement_protocole] Protocole: Osimertinib — T...
--------------------------------------------------

❓ Quels sont les effets secondaires du cisplatine ?
   🔍 Debug: scores={'effets_secondaires': 1} -> best=effets_secondaires
   ✅ Type détecté: effets_secondaires
   🔍 Debug: scores={'ef

In [19]:
class PrioritySmartRetriever(FixedSmartRetriever):
    """Version avec priorité explicite pour effets_secondaires"""
    
    def detect_qtype_smart(self, query: str) -> str:
        """Détection avec priorité aux types spécifiques"""
        q = query.lower()
        
        # PATIENTS SPÉCIFIQUES - PRIORITÉ ABSOLUE
        # Si la question parle d'effets secondaires, c'est TOUJOURS ça
        if re.search(r'effets? secondaires?|effet secondaire|toxicite|toxicité|effets indésirables', q, re.IGNORECASE):
            print(f"   🎯 Priorité: détection 'effets_secondaires' (mot-clé trouvé)")
            return 'effets_secondaires'
        
        if re.search(r'métastase|métastatique|métastases? cérébrales?', q, re.IGNORECASE):
            return 'metastase'
        
        if re.search(r'urgence|compression médullaire|hypercalcémie|aplasie', q, re.IGNORECASE):
            return 'urgence'
        
        if re.search(r'diagnostic|dépistage|comment diagnostiquer|quel examen', q, re.IGNORECASE):
            return 'diagnostic'
        
        if re.search(r'pronostic|survie|récidive|espérance de vie', q, re.IGNORECASE):
            return 'pronostic'
        
        if re.search(r'suivi|surveillance|après traitement', q, re.IGNORECASE):
            return 'suivi'
        
        if re.search(r'résistance|mécanisme de résistance', q, re.IGNORECASE):
            return 'resistance'
        
        # Par défaut: traitement
        return 'traitement'

# Recréer le retriever
priority_retriever = PrioritySmartRetriever(all_chunks, embeddings, encode_query)

print("\n" + "="*80)
print("🧪 TEST AVEC PRIORITÉ CORRIGÉE")
print("="*80)

test_cases = [
    ("Effets secondaires de la chimiothérapie cancer du poumon", "effets_secondaires"),
    ("Quels sont les effets secondaires du cisplatine ?", "effets_secondaires"),
    ("Traitement du cancer du sein métastatique", "traitement"),
    ("Diagnostic du cancer de l'ovaire", "diagnostic"),
    ("Métastases cérébrales du mélanome", "metastase"),
    ("Urgence compression médullaire cancer", "urgence"),
]

for q, expected in test_cases:
    print(f"\n❓ {q}")
    qtype = priority_retriever.detect_qtype_smart(q)
    print(f"   ✅ Détecté: {qtype} | Attendu: {expected}")
    
    if qtype == expected:
        print(f"   🟢 CORRECT")
    else:
        print(f"   🔴 INCORRECT")
    
    results = priority_retriever.search(q, k=3)
    print(f"   📄 Top résultats:")
    for i, r in enumerate(results[:2]):
        icon = "⚠️" if "toxicity" in r['type'] else "🚨" if "emergency" in r['type'] else "💊" if "treatment" in r['type'] else "📄"
        print(f"      {icon} {r['type']} (score: {r['score']:.3f})")
    print("-"*50)

✅ SmartRetriever initialisé (464 chunks)

🧪 TEST AVEC PRIORITÉ CORRIGÉE

❓ Effets secondaires de la chimiothérapie cancer du poumon
   🎯 Priorité: détection 'effets_secondaires' (mot-clé trouvé)
   ✅ Détecté: effets_secondaires | Attendu: effets_secondaires
   🟢 CORRECT
   🎯 Priorité: détection 'effets_secondaires' (mot-clé trouvé)
   📄 Top résultats:
      ⚠️ toxicity_management (score: 2.401)
      ⚠️ toxicity_management (score: 1.881)
--------------------------------------------------

❓ Quels sont les effets secondaires du cisplatine ?
   🎯 Priorité: détection 'effets_secondaires' (mot-clé trouvé)
   ✅ Détecté: effets_secondaires | Attendu: effets_secondaires
   🟢 CORRECT
   🎯 Priorité: détection 'effets_secondaires' (mot-clé trouvé)
   📄 Top résultats:
      ⚠️ toxicity_management (score: 2.137)
      ⚠️ toxicity_management (score: 1.880)
--------------------------------------------------

❓ Traitement du cancer du sein métastatique
   ✅ Détecté: metastase | Attendu: traitement
  

## 6. Prompt adaptatif + pipeline RAG

In [20]:
def format_context_with_citations(results):
    """Formate le contexte avec des références numérotées"""
    context_parts = []
    for i, r in enumerate(results):
        source_ref = f"[Source {i+1}]"
        source_type = r['type'].upper()
        source_cancer = r['cancer']
        header = f"{source_ref} TYPE: {source_type} | CANCER: {source_cancer}"
        context_parts.append(f"{header}\n{r['text']}")
    return "\n\n".join(context_parts)

def build_prompt_with_citations(query, context, qtype, cancer):
    """Construit le prompt en forçant les citations"""
    
    instruction = f"""Tu es un assistant médical expert en oncologie, spécialisé dans {cancer if cancer else 'les cancers'}.

RÈGLES IMPORTANTES:
1. Réponds UNIQUEMENT à partir du contexte fourni
2. CITE TES SOURCES avec des numéros entre crochets [1], [2], etc.
3. Chaque information clé doit avoir une citation
4. Structure ta réponse clairement

Exemple de format attendu:
---
D'après les sources consultées :

- Le traitement de première ligne est X [1]
- Les effets secondaires fréquents incluent Y [2]

📚 Sources:
[1] Protocole de traitement - Cancer prostate
[2] Guide des effets secondaires
---

Contexte fourni:
{context}

Question: {query}

Réponse (avec citations):"""

    return instruction

def rag_generate(query, retriever, k=4):
    """Pipeline RAG complet"""
    
    print(f"\n{'='*70}")
    print(f"🔍 RAG PIPELINE")
    print(f"{'='*70}")
    print(f"❓ Question: {query}")
    
    # 1. Détection
    cancer = retriever.detect_cancer_smart(query)
    qtype = retriever.detect_qtype_smart(query)
    print(f"🎯 Détection: Cancer={cancer}, Type={qtype}")
    
    # 2. Retrieval
    results = retriever.search(query, k=k, cancer_filter=cancer)
    print(f"📚 {len(results)} sources récupérées")
    
    # 3. Affichage des sources
    print(f"\n📄 SOURCES:")
    for i, r in enumerate(results):
        print(f"   [{i+1}] {r['type']} | Score: {r['score']:.3f} | Cancer: {r['cancer']}")
    
    # 4. Construction du prompt
    context = format_context_with_citations(results)
    prompt = build_prompt_with_citations(query, context, qtype, cancer)
    
    # 5. Génération (simulée pour l'instant - à remplacer par vrai LLM)
    answer = f"""D'après les sources consultées sur le {cancer if cancer else 'cancer'} :

**Informations clés :**

{chr(10).join([f"- {r['text'][:150]}... [Source {i+1}]" for i, r in enumerate(results[:3])])}

📚 **Sources documentaires :**
{chr(10).join([f"[{i+1}] {r['type']} - {r['cancer']}" for i, r in enumerate(results)])}

⚠️ Note: Réponse générée à partir de {len(results)} sources. Pour une réponse définitive, connectez un LLM (Phi-3, Gemma, ou Mistral)."""
    
    print(f"\n💬 RÉPONSE GÉNÉRÉE:")
    print(f"{'='*70}")
    print(answer)
    
    return {
        'query': query,
        'cancer': cancer,
        'qtype': qtype,
        'sources': results,
        'answer': answer
    }

# Test du pipeline
test_queries = [
    "Effets secondaires de la chimiothérapie cancer du poumon",
    "Quel est le traitement du cancer de la prostate ?",
    "Diagnostic du cancer de l'ovaire",
]

for q in test_queries:
    result = rag_generate(q, priority_retriever, k=3)
    print("\n" + "="*70)


🔍 RAG PIPELINE
❓ Question: Effets secondaires de la chimiothérapie cancer du poumon
   🎯 Priorité: détection 'effets_secondaires' (mot-clé trouvé)
🎯 Détection: Cancer=Cancer Pulmonaire Non à Petites Cellules (CPNPC), Type=effets_secondaires
   🎯 Priorité: détection 'effets_secondaires' (mot-clé trouvé)
📚 3 sources récupérées

📄 SOURCES:
   [1] toxicity_management | Score: 2.401 | Cancer: Cancer du poumon à petites cellules
   [2] toxicity_management | Score: 1.881 | Cancer: Cancer Pulmonaire Non à Petites Cellules (CPNPC)
   [3] toxicity_management | Score: 1.707 | Cancer: Cancer Pulmonaire Non à Petites Cellules (CPNPC)

💬 RÉPONSE GÉNÉRÉE:
D'après les sources consultées sur le Cancer Pulmonaire Non à Petites Cellules (CPNPC) :

**Informations clés :**

- [Cancer du poumon à petites cellules] [toxicite] Toxicité: Myélosuppression (Neutropénie, Thrombocytopénie). Médicaments: Étoposide; Carboplatine; Cis... [Source 1]
- [Cancer Pulmonaire Non à Petites Cellules (CPNPC)] [toxicite] Toxi

## Évaluation complète des métrique

In [21]:
def evaluate_retrieval_performance(retriever, eval_questions, k_values=[1, 3, 5]):
    """
    Évaluation complète du retrieval
    - Hit@k
    - Recall@k  
    - MRR (Mean Reciprocal Rank)
    """
    
    # Questions de test avec leur type attendu
    eval_set = [
        {"q": "Quel est le traitement du cancer de la prostate ?", "expected_type": "treatment_protocol", "cancer": "Cancer de la prostate"},
        {"q": "Effets secondaires de la chimiothérapie cancer du poumon", "expected_type": "toxicity_management", "cancer": "Cancer Pulmonaire Non à Petites Cellules (CPNPC)"},
        {"q": "Comment diagnostiquer le cancer de l'ovaire ?", "expected_type": "diagnostic_guideline", "cancer": "Cancer épithélial de l'ovaire"},
        {"q": "Métastases cérébrales du mélanome", "expected_type": "metastasis", "cancer": "Mélanome"},
        {"q": "Urgence compression médullaire cancer", "expected_type": "oncology_emergency", "cancer": None},
        {"q": "Stadification du cancer du sein", "expected_type": "staging_system", "cancer": "Cancer du sein"},
        {"q": "Suivi après cancer de la thyroïde", "expected_type": "followup", "cancer": "Cancer de la thyroïde"},
        {"q": "Résistance au trastuzumab cancer du sein", "expected_type": "resistance_mechanism", "cancer": "Cancer du sein"},
    ]
    
    print("="*80)
    print("📊 ÉVALUATION DES PERFORMANCES DU RETRIEVER")
    print("="*80)
    
    results_summary = {}
    
    for k in k_values:
        results_summary[k] = {"hits": 0, "recalls": [], "reciprocal_ranks": []}
    
    for item in eval_set:
        query = item["q"]
        expected = item["expected_type"]
        cancer = item.get("cancer")
        
        print(f"\n❓ {query}")
        print(f"   Attendu: {expected}")
        
        # Recherche avec k=max(k_values)
        results = retriever.search(query, k=max(k_values), cancer_filter=cancer)
        found_types = [r["type"] for r in results]
        
        # Trouver le rang du résultat attendu
        rank = None
        for i, t in enumerate(found_types):
            if t == expected:
                rank = i + 1
                break
        
        # Calcul pour chaque k
        for k in k_values:
            top_k = found_types[:k]
            is_hit = expected in top_k
            
            if is_hit:
                results_summary[k]["hits"] += 1
            
            # Recall@k (binaire: 1 si trouvé dans top-k, sinon 0)
            results_summary[k]["recalls"].append(1 if is_hit else 0)
            
            # MRR
            if rank and rank <= k:
                results_summary[k]["reciprocal_ranks"].append(1/rank)
            elif k == max(k_values):
                results_summary[k]["reciprocal_ranks"].append(0)
        
        # Affichage
        if rank:
            print(f"   ✅ Trouvé au rang {rank}")
            print(f"   📊 Top-3: {found_types[:3]}")
        else:
            print(f"   ❌ Non trouvé dans top-{max(k_values)}")
            print(f"   📊 Top-3: {found_types[:3]}")
    
    # Affichage des résultats agrégés
    print("\n" + "="*80)
    print("📈 RÉSULTATS AGRÉGÉS")
    print("="*80)
    
    n_queries = len(eval_set)
    
    for k in k_values:
        hit_rate = results_summary[k]["hits"] / n_queries
        recall = np.mean(results_summary[k]["recalls"])
        mrr = np.mean(results_summary[k]["reciprocal_ranks"])
        
        print(f"\n🔹 Hit@{k}: {results_summary[k]['hits']}/{n_queries} = {hit_rate*100:.1f}%")
        print(f"🔹 Recall@{k}: {recall*100:.1f}%")
        print(f"🔹 MRR@{k}: {mrr:.3f}")
    
    print("\n" + "="*80)
    print("🎯 SCORE GLOBAL (Hit@3) : référence pour la soutenance")
    print(f"   {results_summary[3]['hits']}/{n_queries} = {results_summary[3]['hits']/n_queries*100:.1f}%")
    print("="*80)
    
    return results_summary

# Exécution de l'évaluation
print("\n🚀 LANCEMENT DE L'ÉVALUATION...\n")
eval_results = evaluate_retrieval_performance(priority_retriever, None, k_values=[1, 3, 5])


🚀 LANCEMENT DE L'ÉVALUATION...

📊 ÉVALUATION DES PERFORMANCES DU RETRIEVER

❓ Quel est le traitement du cancer de la prostate ?
   Attendu: treatment_protocol
   ✅ Trouvé au rang 1
   📊 Top-3: ['treatment_protocol', 'treatment_protocol', 'treatment_protocol']

❓ Effets secondaires de la chimiothérapie cancer du poumon
   Attendu: toxicity_management
   🎯 Priorité: détection 'effets_secondaires' (mot-clé trouvé)
   ✅ Trouvé au rang 1
   📊 Top-3: ['toxicity_management', 'toxicity_management', 'toxicity_management']

❓ Comment diagnostiquer le cancer de l'ovaire ?
   Attendu: diagnostic_guideline
   ❌ Non trouvé dans top-5
   📊 Top-3: ['treatment_protocol', 'metastasis', 'metastasis']

❓ Métastases cérébrales du mélanome
   Attendu: metastasis
   ✅ Trouvé au rang 1
   📊 Top-3: ['metastasis', 'followup', 'followup']

❓ Urgence compression médullaire cancer
   Attendu: oncology_emergency
   ✅ Trouvé au rang 1
   📊 Top-3: ['oncology_emergency', 'oncology_emergency', 'oncology_emergency']

❓

## 7. LLMS

In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
import gc

class MultiLLM:
    """Gestionnaire de 3 LLM locaux pour benchmark"""
    
    def __init__(self):
        self.models = {}
        self.tokenizers = {}
        self.device = "cpu"
        print(f"🖥️ Device: {self.device}")
        print(f"⚠️ Chargement de 3 modèles sur CPU peut prendre 10-15 minutes...")
    
    def load_tinyllama(self):
        """TinyLlama - 1.1B paramètres (plus léger)"""
        print("\n🔄 Chargement TinyLlama (1.1B)...")
        try:
            model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
            self.tokenizers['tinyllama'] = AutoTokenizer.from_pretrained(model_name)
            self.models['tinyllama'] = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float32,
                device_map="cpu",
                low_cpu_mem_usage=True
            )
            self.models['tinyllama'].eval()
            print(f"✅ TinyLlama chargé")
            return True
        except Exception as e:
            print(f"❌ Erreur TinyLlama: {e}")
            return False
    
    def load_phi2(self):
        """Phi-2 - 2.7B paramètres (Microsoft)"""
        print("\n🔄 Chargement Phi-2 (2.7B)...")
        try:
            model_name = "microsoft/phi-2"
            self.tokenizers['phi2'] = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            self.models['phi2'] = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float32,
                device_map="cpu",
                trust_remote_code=True,
                low_cpu_mem_usage=True
            )
            self.models['phi2'].eval()
            print(f"✅ Phi-2 chargé")
            return True
        except Exception as e:
            print(f"❌ Erreur Phi-2: {e}")
            return False
    
    def load_gemma_2b(self):
        """Gemma-2B - 2B paramètres (Google) - version sans authentification"""
        print("\n🔄 Chargement Gemma-2B (2B)...")
        try:
            # Utiliser une version mirror ou quantisée
            model_name = "google/gemma-2b"
            self.tokenizers['gemma'] = AutoTokenizer.from_pretrained(model_name)
            self.models['gemma'] = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float32,
                device_map="cpu",
                low_cpu_mem_usage=True
            )
            self.models['gemma'].eval()
            print(f"✅ Gemma-2B chargé")
            return True
        except Exception as e:
            print(f"❌ Erreur Gemma: {e}")
            print("   Alternative: Utilisation de Qwen2-1.5B à la place")
            return self.load_qwen2()
    
    def load_qwen2(self):
        """Qwen2-1.5B (alternative à Gemma)"""
        print("\n🔄 Chargement Qwen2-1.5B (alternative)...")
        try:
            model_name = "Qwen/Qwen2-1.5B-Instruct"
            self.tokenizers['qwen2'] = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            self.models['qwen2'] = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float32,
                device_map="cpu",
                trust_remote_code=True,
                low_cpu_mem_usage=True
            )
            self.models['qwen2'].eval()
            print(f"✅ Qwen2-1.5B chargé")
            return True
        except Exception as e:
            print(f"❌ Erreur Qwen2: {e}")
            return False
    
    def generate(self, model_name, prompt, max_new_tokens=200):
        """Génération avec un modèle spécifique"""
        if model_name not in self.models:
            return f"❌ Modèle {model_name} non chargé", 0
        
        model = self.models[model_name]
        tokenizer = self.tokenizers[model_name]
        
        # Formatage du prompt
        if model_name == "tinyllama":
            formatted_prompt = f"<|system|>\nTu es un expert en oncologie.</s>\n<|user|>\n{prompt}</s>\n<|assistant|>\n"
        elif model_name == "phi2":
            formatted_prompt = f"Instruct: {prompt}\nOutput:"
        elif model_name in ["gemma", "qwen2"]:
            formatted_prompt = f"<|im_start|>system\nTu es un expert en oncologie.<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
        else:
            formatted_prompt = prompt
        
        inputs = tokenizer(formatted_prompt, return_tensors="pt", truncation=True, max_length=1500)
        
        start_time = time.time()
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.3,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )
        
        latency = time.time() - start_time
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        return response, latency

# Initialisation et chargement des 3 modèles
multillm = MultiLLM()

print("\n" + "="*70)
print("🚀 CHARGEMENT DE 3 LLM LOCAUX")
print("="*70)

# Charger les 3 modèles
models_loaded = {}

# 1. TinyLlama (déjà chargé)
print("\n📦 [1/3] TinyLlama...")
models_loaded['tinyllama'] = multillm.load_tinyllama()

# 2. Phi-2
print("\n📦 [2/3] Phi-2...")
models_loaded['phi2'] = multillm.load_phi2()

# 3. Qwen2 (remplacement Gemma)
print("\n📦 [3/3] Qwen2-1.5B...")
models_loaded['qwen2'] = multillm.load_qwen2()

print("\n" + "="*70)
print("📊 STATUT DES MODÈLES")
print("="*70)
for name, status in models_loaded.items():
    print(f"{name:15} : {'✅ CHARGÉ' if status else '❌ NON CHARGÉ'}")

# Compter les modèles chargés
loaded_count = sum(models_loaded.values())
print(f"\n✅ {loaded_count}/3 modèles chargés avec succès")

🖥️ Device: cpu
⚠️ Chargement de 3 modèles sur CPU peut prendre 10-15 minutes...

🚀 CHARGEMENT DE 3 LLM LOCAUX

📦 [1/3] TinyLlama...

🔄 Chargement TinyLlama (1.1B)...


Loading weights: 100%|██████████| 201/201 [00:10<00:00, 19.73it/s]


✅ TinyLlama chargé

📦 [2/3] Phi-2...

🔄 Chargement Phi-2 (2.7B)...


Loading weights: 100%|██████████| 453/453 [01:08<00:00,  6.60it/s]


✅ Phi-2 chargé

📦 [3/3] Qwen2-1.5B...

🔄 Chargement Qwen2-1.5B (alternative)...


Loading weights: 100%|██████████| 338/338 [00:03<00:00, 87.60it/s] 


✅ Qwen2-1.5B chargé

📊 STATUT DES MODÈLES
tinyllama       : ✅ CHARGÉ
phi2            : ✅ CHARGÉ
qwen2           : ✅ CHARGÉ

✅ 3/3 modèles chargés avec succès


## Benchmark comparatif des 3 LLM

In [26]:
def benchmark_llms(query, retriever, multillm, models_to_test, k=2):
    """Compare les performances des 3 LLM"""
    
    print("\n" + "="*80)
    print(f"🏆 BENCHMARK COMPARATIF DES LLM LOCAUX")
    print("="*80)
    print(f"❓ Question: {query}\n")
    
    # Récupération du contexte (identique pour tous)
    cancer = retriever.detect_cancer_smart(query)
    results = retriever.search(query, k=k, cancer_filter=cancer)
    
    context_parts = []
    for i, r in enumerate(results):
        context_parts.append(f"[Source {i+1}] {r['type']}\n{r['text'][:400]}")
    context = "\n\n".join(context_parts)
    
    prompt = f"""CONTEXTE MÉDICAL:
{context}

QUESTION: {query}

Réponds précisément en citant les sources [1], [2]. Si l'information n'est pas dans le contexte, dis-le."""
    
    # Tester chaque modèle
    benchmark_results = []
    
    for model_name in models_to_test:
        if model_name not in multillm.models:
            continue
            
        print(f"\n{'='*60}")
        print(f"🤖 TEST: {model_name.upper()}")
        print(f"{'='*60}")
        
        try:
            response, latency = multillm.generate(model_name, prompt, max_new_tokens=250)
            
            benchmark_results.append({
                'Modèle': model_name.upper(),
                'Latence (s)': round(latency, 2),
                'Taille': '1.1B' if model_name == 'tinyllama' else '2.7B' if model_name == 'phi2' else '1.5B',
                'Réponse': response[:200] + "...",
                'Qualité': '⭐' * min(3, int(200/latency)) if latency > 0 else '⭐'
            })
            
            print(f"⏱️ Temps: {latency:.2f}s")
            print(f"📝 Réponse: {response[:300]}...")
            
        except Exception as e:
            print(f"❌ Erreur: {e}")
    
    # Tableau comparatif
    print("\n" + "="*80)
    print("📊 TABLEAU COMPARATIF DES PERFORMANCES")
    print("="*80)
    
    df = pd.DataFrame(benchmark_results)
    if not df.empty:
        print(df.to_string(index=False))
    
    # Conclusion
    print("\n" + "="*80)
    print("🎯 CONCLUSION DU BENCHMARK")
    print("="*80)
    
    if benchmark_results:
        # Meilleur temps
        fastest = min(benchmark_results, key=lambda x: x['Latence (s)'])
        print(f"⚡ Plus rapide: {fastest['Modèle']} ({fastest['Latence (s)']}s)")
        
        # Recommandation
        print(f"""
📌 RECOMMANDATION POUR LA SOUTENANCE:
--------------------------------------------------------------------------------
• TinyLlama  : Léger mais moins précis - OK pour démo
• Phi-2      : Bon compromis qualité/vitesse - RECOMMANDÉ ⭐
• Qwen2-1.5B : Bonne qualité mais plus lent

✅ Meilleur choix: Phi-2 (Microsoft) - 2.7B paramètres
   - Bonne compréhension du français
   - Rapide sur CPU (~30-45s)
   - Spécialisé pour les instructions
""")
    
    return benchmark_results

# Exécution du benchmark si au moins 2 modèles chargés
if loaded_count >= 2:
    # Utiliser uniquement les modèles chargés
    available_models = [name for name, status in models_loaded.items() if status]
    
    # Question de test médicale
    test_query = "Quels sont les effets secondaires du cisplatine en chimiothérapie ?"
    
    benchmark_results = benchmark_llms(
        test_query, 
        priority_retriever, 
        multillm, 
        available_models,
        k=2
    )
else:
    print("\n⚠️ Pas assez de modèles chargés pour un benchmark significatif.")
    print("   Minimum requis: 2 modèles sur 3")


🏆 BENCHMARK COMPARATIF DES LLM LOCAUX
❓ Question: Quels sont les effets secondaires du cisplatine en chimiothérapie ?

   🎯 Priorité: détection 'effets_secondaires' (mot-clé trouvé)

🤖 TEST: TINYLLAMA


Both `max_new_tokens` (=250) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⏱️ Temps: 118.08s
📝 Réponse: CONTEXTE MÉDICAL:
[Source 1] toxicity_management
[Cancer du col de l'utérus] [toxicite] Toxicité: Néphrotoxicité induite par le Cisplatine. Médicaments: Cisplatine. Gestion: Grade_1: Maintenir l'intensité de dose du cisplatine sous étroite surveillance biologique.; Optimiser l'hydra | Grade_2: Inter...

🤖 TEST: PHI2
⏱️ Temps: 414.07s
📝 Réponse:  The second effects of cisplatin in chemotherapy are:

- Nephrotoxicity, induced by cisplatin, is a toxicity that affects the kidneys.
- Neuropathy, a peripheral neuropathy, is a toxicity that affects the nerves.
- Cisplatin can cause cumulative toxicity, meaning that the dose accumulates over time ...

🤖 TEST: QWEN2
⏱️ Temps: 174.74s
📝 Réponse: Les effets secondaires du cisplatine en chimiothérapie peuvent inclure:

- Néphrotoxicité: La toxicité néphrotique est un effet secondaire courant du cisplatine.
- Toxicité nerveuse: Il peut également entraîner des troubles nerveux tels que la neuropathie périphérique et la m